# E-Commerce Sales Analysis — Olist (Brazil, 2016–2018)
End-to-end data analyst project: **SQL → Python → Power BI**.

**Business questions:**
1. What are the headline KPIs (revenue, orders, AOV)?
2. How is revenue trending month over month?
3. Which product categories drive revenue — and which drag?
4. How reliable is delivery, and does lateness hurt satisfaction?
5. How do customers pay, and how do installments scale with basket size?
6. Where does freight eat the margin?

_Dataset: Olist public data, ~100k orders (CC BY-SA 4.0)._


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
RAW = '../data/raw'  # run from python/, or adjust


## 1. Load & profile the raw data


In [ ]:
orders = pd.read_csv(f'{RAW}/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'])
items  = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
pay    = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
rev    = pd.read_csv(f'{RAW}/olist_order_reviews_dataset.csv')
cust   = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
prod   = pd.read_csv(f'{RAW}/olist_products_dataset.csv')
for name, df in [('orders',orders),('items',items),('payments',pay),('reviews',rev),('customers',cust),('products',prod)]:
    print(f'{name:9s} rows={len(df):,}  nulls={int(df.isna().sum().sum()):,}  dup_rows={int(df.duplicated().sum()):,}')


## 2. Data cleaning
Issues found during profiling and how each is handled (same logic as `sql/02_data_cleaning.sql`):
- **166 orders** have `order_delivered_carrier_date` *before* the purchase timestamp (logging error) → set carrier date to NULL, keep the order.
- **814 duplicate `review_id`s** → keep the latest review per id.
- **9 payments** with `payment_value <= 0` → excluded (not real money movement).
- **610 products** missing category → labeled `'unknown'` (kept for revenue math); 2 missing dimensions left NULL (missing ≠ zero).
- NULL delivery dates on canceled/in-flight orders are legitimate → kept.


In [ ]:
bad = (orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']).fillna(False)
print('carrier-date-before-purchase rows nulled:', int(bad.sum()))
orders.loc[bad, 'order_delivered_carrier_date'] = pd.NaT
rev = rev.sort_values('review_creation_date').drop_duplicates('review_id', keep='last')
print('reviews after dedup:', f'{len(rev):,}')
pay = pay[pay['payment_value'] > 0]
prod['product_category_name'] = prod['product_category_name'].fillna('unknown')
tr = pd.read_csv(f'{RAW}/product_category_name_translation.csv')
cat = dict(zip(tr['product_category_name'], tr['product_category_name_english']))
prod['category_en'] = prod['product_category_name'].map(lambda c: cat.get(c, c))


## 3. Headline KPIs (delivered orders = recognized revenue)


In [ ]:
d = orders[orders.order_status == 'delivered']
di = items[items.order_id.isin(set(d.order_id))]
rev_total = di['price'].sum(); freight_total = di['freight_value'].sum()
aov = di.groupby('order_id')['price'].sum().mean()
print(f"Delivered orders : {len(d):,}")
print(f"Revenue          : R$ {rev_total:,.0f}")
print(f"Freight cost     : R$ {freight_total:,.0f}")
print(f"Average order value: R$ {aov:,.2f}")
print(f"Unique customers : {d.merge(cust,on='customer_id')['customer_unique_id'].nunique():,}")


## 4. Revenue trend — is the business growing?


In [ ]:
m = d.copy(); m['ym'] = m['order_purchase_timestamp'].dt.to_period('M')
trend = m.groupby('ym').agg(orders=('order_id','nunique'))
trend['revenue'] = trend.index.map(lambda p: di[items['order_id'].isin(set(m[m['ym']==p]['order_id']))]['price'].sum())
trend['mom_pct'] = trend['revenue'].pct_change()*100
ax = trend['revenue'].plot(figsize=(10,4), title='Monthly revenue (delivered orders)')
ax.set_ylabel('BRL'); plt.tight_layout(); plt.show()
print(trend.tail(6).round(1).to_string())


## 5. Category performance — winners and losers


In [ ]:
g = di.merge(prod[['product_id','category_en']], on='product_id').groupby('category_en').agg(revenue=('price','sum'), orders=('order_id','nunique')).sort_values('revenue', ascending=False)
print('Top 5:'); print(g.head(5).round(0).to_string())
print('\nBottom 5:'); print(g.tail(5).round(0).to_string())
g.head(10)['revenue'].sort_values().plot.barh(figsize=(8,5), title='Top 10 categories by revenue (BRL)')
plt.tight_layout(); plt.show()


## 6. Delivery reliability — the 8% problem
Late = delivered after the estimated delivery date.


In [ ]:
dd = d.dropna(subset=['order_delivered_customer_date']).copy()
dd['late'] = dd['order_delivered_customer_date'] > dd['order_estimated_delivery_date']
dd['days'] = (dd['order_delivered_customer_date'] - dd['order_purchase_timestamp']).dt.days
print(f"Avg delivery: {dd['days'].mean():.1f} days (median {dd['days'].median():.0f})")
print(f"Late deliveries: {dd['late'].mean():.1%} ({int(dd['late'].sum()):,} orders)")
st = dd.merge(cust[['customer_id','customer_state']], on='customer_id').groupby('customer_state').agg(o=('order_id','count'), late=('late','mean'), days=('days','mean'))
print(st[st['o']>=500].sort_values('late', ascending=False).head(5).round(3).to_string())


## 7. Does late delivery hurt satisfaction?


In [ ]:
r = rev.merge(d[['order_id']], on='order_id').merge(dd[['order_id','late']], on='order_id')
print(r.groupby('late')['review_score'].agg(['mean','count']).round(2).to_string())
r.groupby('late')['review_score'].value_counts(normalize=True).unstack().plot.bar(figsize=(8,4), title='Review score distribution: on-time vs late')
plt.tight_layout(); plt.show()


## 8. Payments — installments scale with basket size


In [ ]:
print(pay['payment_type'].value_counts(normalize=True).round(3).to_string())
po = pay.groupby('order_id').agg(v=('payment_value','sum'), inst=('payment_installments','max')).reset_index()
po['band'] = pd.cut(po['v'], [0,100,300,float('inf')], labels=['<100','100–300','300+'])
print(po.groupby('band', observed=True)['inst'].mean().round(2).to_string())


## 9. Freight burden by category (weighted: total freight / total revenue)


In [ ]:
f = di.merge(prod[['product_id','category_en']], on='product_id')
fb = f.groupby('category_en').agg(n=('price','count'), burden=('price', lambda s: f.loc[s.index,'freight_value'].sum()/s.sum()*100))
print(fb.query('n>=500').sort_values('burden', ascending=False).head(8).round(1).to_string())


## 10. Export the cleaned fact table for Power BI
One denormalized row per order item → `powerbi/fact_orders_clean.csv`.


In [ ]:
# (fact table built by python/build_notebook.py -> FACT_PATH)
print('See powerbi/fact_orders_clean.csv and powerbi/POWERBI_GUIDE.md')


## Key findings & recommendations
1. **R$13.2M revenue** across 96.5k delivered orders (AOV R$137); top 3 categories (health & beauty, watches & gifts, bed/bath/table) drive ~26% of revenue.
2. **8.1% of orders arrive late**; late orders score **2.6/5 vs 4.3/5** on-time — delivery reliability is the biggest lever on customer satisfaction.
3. **Late delivery concentrates in the North/Northeast** (MA 19.7%, CE 15.3% vs ~8% average) with 21+ day average transits → fix the long-haul lanes first.
4. **Credit card = 74% of payments**; installments rise from 2.0× (<R$100) to 5.3× (R$300+) → promote split payments on high baskets to lift conversion.
5. **Freight burden is heaviest on bulky goods** (electronics ~30%, furniture ~24–25% of revenue) → renegotiate carrier rates or set free-shipping thresholds by category.
6. **Repeat purchase rate is only 3%** and 66% of orders come from SP/RJ/MG → retention campaigns and geographic expansion are the growth levers.
